### **UTILIZATION OF LSTM ON MNIST DATASET**

In [ ]:
# imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

# tensorflow and scikit learn imports
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

In [ ]:
# loading the dataset

train_path = pd.read_csv("dataset/mnist_train.csv")
test_path = pd.read_csv("dataset/mnist_test.csv")

print("Dataset Loaded Successfully!")

print(f"Train Shape: {train_path.shape}")
print(f"Test Shape: {test_path.shape}")
print(f"First 5 records of the train dataset:\n{train_path.head()}")

print(f"Train CSV Columns: {train_path.columns.tolist()}")
print(f"Test CSV Columns: {test_path.columns.tolist()}")

In [ ]:
# preprocessing train set

X = train_path.iloc[:, 1:].values / 255.0
y = train_path.iloc[:, 0].values

# reshape → RGB → resize
X = X.reshape(-1, 28, 28, 1)
X = np.repeat(X, 3, axis=-1)
X = np.array([cv2.resize(img, (224,224)) for img in X])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

y_train = to_categorical(y_train, 10)
y_val   = to_categorical(y_val, 10)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")

# preprocessing test set

X_test = test_path.iloc[:, 1:].values / 255.0
X_test = X_test.reshape(-1, 28, 28, 1)
X_test = np.repeat(X_test, 3, axis=-1)
X_test = np.array([cv2.resize(img, (224,224)) for img in X_test])

print(f"X_test: {X_test.shape}")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# building the model

def build_resnet_model():
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )

    for layer in base_model.layers:
        layer.trainable = False

    x = Flatten()(base_model.output)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(10, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

model = build_resnet_model()
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
# === NO AUGMENTATION MODEL ===

print("Training started for MODEL with NO AUGMENTATION!")
no_aug_model = build_resnet_model()
no_aug_history = no_aug_model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

# === AUGMENTATION MODEL ===

print("Training started for MODEL with AUGMENTATION!")

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

datagen.fit(X_train)

augmented_model = build_resnet_model()
augmented_history = augmented_model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=5,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
plt.figure(figsize=(12,10))

# Augmented Accuracy
plt.subplot(2,2,1)
plt.plot(augmented_history.history['accuracy'], label='Train Accuracy')
plt.plot(augmented_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model with Augmentation - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Augmented Loss
plt.subplot(2,2,2)
plt.plot(augmented_history.history['loss'], label='Train Loss')
plt.plot(augmented_history.history['val_loss'], label='Validation Loss')
plt.title('Model with Augmentation - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# No Aug Accuracy
plt.subplot(2,2,3)
plt.plot(no_aug_history.history['accuracy'], label='Train Accuracy')
plt.plot(no_aug_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model with No Augmentation - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# No Aug Loss
plt.subplot(2,2,4)
plt.plot(no_aug_history.history['loss'], label='Train Loss')
plt.plot(no_aug_history.history['val_loss'], label='Validation Loss')
plt.title('Model with No Augmentation - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# test prediction using resnet50 Model

preds = augmented_model.predict(X_test)
y_pred = np.argmax(preds, axis=1)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(X_test)+1),
    "Label": y_pred
})

submission.to_csv("resnet50_submission.csv", index=False)
print(submission.head())